# Initialization

Files for case studies and meteorological station data are available on [GitHub](https://github.com/thomaslu678/Praxis-Lab-25-26/tree/main/assets). These assets should be uploaded to an individual Google Earth Engine project for further access.

## Expected Feature Columns
### Studies

* name (str)
* start (int)
* end (int)
* similarity (float)
* station (str)

### Meterological Data

* STATION (str)
* Year (int)
* Month (int)
* Day (int)
* Hour (int)
* temperature (float)
* relative_humidity (float)

In [ ]:
import ee, requests, geemap.core as geemap
from google.colab import userdata

# Retrieve project-specific info as Colab secrets
project_name = userdata.get("project_name")
studies_path = userdata.get("studies_path")
met_path = userdata.get("met_path")

export_path = userdata.get("export_path")


ee.Authenticate()
ee.Initialize(project = project_name)

# Store studies as client-side dictionary of server-side features
# (should be ok due to small number of case studies)
study_collection = ee.FeatureCollection(studies_path)

studies = {
    study["properties"]["name"].replace(" ", ""):
    ee.Feature(study)
    for study
    in study_collection.toList(study_collection.size()).getInfo()
}

met_data = ee.FeatureCollection(met_path)

crs = ee.Projection("EPSG:4326").atScale(30)

# Combine Landsat data into single dataset

Includes Landsat 9, 8, 7, 5, and 4

In [ ]:
ls9: ee.ImageCollection = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
ls9 = ls9.select([
    "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7", "ST_B10", "QA_PIXEL", "QA_RADSAT"
], [
    "blue", "green", "red", "nir", "swir1", "swir2", "lst", "QA_PIXEL", "QA_RADSAT"
])

ls8: ee.ImageCollection = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
ls8 = ls8.select([
    "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7", "ST_B10", "QA_PIXEL", "QA_RADSAT"
], [
    "blue", "green", "red", "nir", "swir1", "swir2", "lst", "QA_PIXEL", "QA_RADSAT"
])

ls7: ee.ImageCollection = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
ls7 = ls7.select([
    "SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7", "ST_B6", "QA_PIXEL", "QA_RADSAT"
], [
    "blue", "green", "red", "nir", "swir1", "swir2", "lst", "QA_PIXEL", "QA_RADSAT"
])

ls5: ee.ImageCollection = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
ls5 = ls5.select([
    "SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7", "ST_B6", "QA_PIXEL", "QA_RADSAT"
], [
    "blue", "green", "red", "nir", "swir1", "swir2", "lst", "QA_PIXEL", "QA_RADSAT"
])

ls4: ee.ImageCollection = ee.ImageCollection("LANDSAT/LT04/C02/T1_L2")
ls4 = ls4.select([
    "SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7", "ST_B6", "QA_PIXEL", "QA_RADSAT"
], [
    "blue", "green", "red", "nir", "swir1", "swir2", "lst", "QA_PIXEL", "QA_RADSAT"
])

all_landsat: ee.ImageCollection = ls9.merge(ls8).merge(ls7).merge(ls5).merge(ls4)

# Further filtering and processing of Landsat data

* Remove images from non-summer months
  * Also remove meteorological data for non-summer months
* Mask bad pixels
  * Clouds
  * Water
  * Saturated pixels
* Transform digital numbers to real values
* Encode time as individual band

In [ ]:
# Only include data from June, July, and August
landsat: ee.ImageCollection = all_landsat.filter(
    ee.Filter.calendarRange(6, 8, "month")
)

# Only include met data from June, July, and August
met_data = met_data.filter(
    ee.Filter.rangeContains("Month", 6, 8)
)

# Mask pixels with clouds or water, or saturated pixels
def apply_mask(image: ee.Image) -> ee.Image:
    # Bit 0: Fill
    # Bit 1: Dilated Cloud
    # Bit 2: Cirrus (high confidence)
    # Bit 3: Cloud
    # Bit 4: Cloud Shadow
    # Bit 7: Water
    qa_mask = image.select("QA_PIXEL").bitwiseAnd(int("10011111", 2)).eq(0)

    saturation_mask = image.select("QA_RADSAT").eq(0)

    return image.updateMask(qa_mask).updateMask(saturation_mask)

landsat = landsat.map(apply_mask)

# Scale Landsat data to useful measurements
def scale_landsat(image: ee.Image) -> ee.Image:
    optical_bands = image.select("blue", "green", "red", "nir", "swir1", "swir2").multiply(0.0000275).add(-0.2)
    thermal_bands = image.select("lst").multiply(0.00341802).add(149.0)

    return image.addBands(optical_bands, None, True).addBands(
        thermal_bands,
        None,
        True
    )

landsat = landsat.map(scale_landsat)

# Encode time
landsat = landsat.map(
    lambda image: image.addBands(
        ee.Image.constant(image.getNumber("system:time_start"))
        .rename("time")
        .int64()
    )
)

# Remove QA bands
landsat = landsat.select(
    landsat.first().bandNames().filter(
        ee.Filter.And(
            ee.Filter.neq("item", "QA_PIXEL"),
            ee.Filter.neq("item", "QA_RADSAT")
        )
    )
)

# Generate Samples

Every case study is associated with a set of transect points. Every transect point is associated with several samples in time. Each sample contains relevant data for a specific point in time.

## Output Data Columns

* study_id (int)
* start (int)
* end (int)
* similarity (float)
* distance (int)
* time (int)
* station_temp (float)
* station_humidity (float)
* blue (float)
* green (float)
* red (float)
* nir (float)
* swir1 (float)
* swir2 (float)

In [ ]:
def generate_samples(study: ee.Feature) -> ee.List:

  # Only use Landsat images within 5 years before and after case study
  images: ee.ImageCollection = landsat.filter(
      ee.Filter.calendarRange(
          study.getNumber("start").add(-5),
          study.getNumber("end").add(5),
          "year"
      )
  )

  # Only include Landsat images within 300 meters from case study
  buffer = study.geometry().buffer(300)
  images = images.filterBounds(buffer)

  # Calculate pixel distances from case study
  distances = (
      ee.Image.constant(1)
      .clip(study)
      .mask()
      .fastDistanceTransform(32, metric="manhattan")
      .reproject(crs)
  )

  # For each image:
  # add new bands - study_id, similarity, start, end, distance, station_temp, station_humidity
  # mask pixels on case study (distance = 0 or distance > 32)
  # reproject to match distance raster

  mask = distances.lt(32).bitwiseAnd(
      distances.gt(0)
  )

  station_data: ee.FeatureCollection = met_data.filter(
      ee.Filter.eq("STATION", study.get("station"))
  )

  def process_image(image: ee.Image) -> ee.Image:

    # Met data
    time = image.date()
    year = time.get("year")
    month = time.get("month")
    day = time.get("day")
    hour = time.get("hour")

    met_data: ee.Feature = station_data.filter(
        ee.Filter.And(
            ee.Filter.eq("Year", year),
            ee.Filter.eq("Month", month),
            ee.Filter.eq("Day", day),
            ee.Filter.eq("Hour", hour),
        )
    ).first()

    # Fallback to -9999 if no met data available
    station_temp = ee.Algorithms.If(
        met_data,
        met_data.getNumber("temperature"),
        -9999
    )

    station_humidity = ee.Algorithms.If(
        met_data,
        met_data.getNumber("relative_humidity"),
        -9999
    )

    return image.addBands(
        ee.Image.constant(study.getNumber("id"))
        .rename("study_id")
        .int16()
    ).addBands(
        ee.Image.constant(study.getNumber("start"))
        .rename("start")
        .int16()
    ).addBands(
        ee.Image.constant(study.getNumber("end"))
        .rename("end")
        .int16()
    ).addBands(
        ee.Image.constant(station_temp)
        .rename("station_temp")
        .float()
    ).addBands(
        ee.Image.constant(station_humidity)
        .rename("station_humidity")
        .float()
    ).addBands(
        distances
    ).updateMask(
        mask
    ).reproject(
        crs
    )

  images = images.map(process_image)


  # Sample all remaining pixels
  samples = images.map(
      lambda image: image.sample(
          buffer,
          scale = crs.nominalScale(),
          projection = crs,
      )
  )

  return samples.flatten().filter(
      ee.Filter.And(
          ee.Filter.neq("station_temp", -9999),
          ee.Filter.neq("station_humidity", -9999)
      )
  )

# Run function to generate samples
samples = {
    k: generate_samples(v) for k, v in studies.items()
}

# Export

Data for each case study will be individually exported as an Earth Engine asset.

In [ ]:
# Store tasks in dictionary for potential future querying
tasks = {
    k: ee.batch.Export.table.toAsset(
        collection = v,
        assetId = export_path + k,
        description = k
    ) for k, v in samples.items()
}

# Start all tasks
for v in tasks.values():
  v.start()
  print(v.status())

{'state': 'READY', 'description': 'ParcLineairedelaRiviereSaint-Charles', 'priority': 100, 'creation_timestamp_ms': 1775797273110, 'update_timestamp_ms': 1775797273110, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'UYGAWQFG4MWXCV26P2NUBQIG', 'name': 'projects/gee-481701/operations/UYGAWQFG4MWXCV26P2NUBQIG'}
{'state': 'READY', 'description': 'MadridRio', 'priority': 100, 'creation_timestamp_ms': 1775797275069, 'update_timestamp_ms': 1775797275069, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'KF6O7QME4NKTJHSGDXCTK3B3', 'name': 'projects/gee-481701/operations/KF6O7QME4NKTJHSGDXCTK3B3'}
{'state': 'READY', 'description': 'AtlantaBeltline', 'priority': 100, 'creation_timestamp_ms': 1775797275928, 'update_timestamp_ms': 1775797275928, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'IRJYSDITP75S3VORHF6LQX3R', 'name': 'projects/gee-481701/operations/IRJYSDITP75S3VORHF6LQX3R'}
{'state': 'READY', 'description': 'AnacostiaRiverwalkTrail', 'prior

In case you want to cancel everything...

In [ ]:
for v in tasks.values():
  v.cancel()

# Merging

In [ ]:
study_names = map(
    lambda study: study["properties"]["name"].replace(" ", ""),
    studies
)

sample_list = list(map(
    lambda name: ee.FeatureCollection(export_path + name),
    study_names
))

final = ee.FeatureCollection(sample_list).flatten()

task = ee.batch.Export.table.toAsset(
    collection = final,
    assetId = export_path + "Merged",
    description = "Merged studies"
)
task.start()

print(task.status())